الهدف: تدريب مرمز صور خام (Unfiltered Image Tokenizer) على بيانات بكسلية متحررة من الفلاتر لضمان القدرة على ضغط وتشفير كافة أنماط الصور، وإنتاج الملف الأساسي image_tokenizer.pt اللازم لدفتر الفيديو.

الخلية (0): تهيئة البيئة ورفع القيود وإلغاء فلاتر الفحص المسبق

In [ ]:
import resource
import sys
import os

# رفع حدود الموارد والملفات المفتوحة إلى الحد الأقصى
try:
    resource.setrlimit(resource.RLIMIT_NOFILE, (65536, 65536))
    print("تم رفع قيود عدد الملفات المفتوحة بنجاح إلى 65536")
except Exception as e:
    print(f"تنبيه عند رفع القيود: {e}")

# ضبط بيئة التنفيذ وإلغاء فلاتر الأمان والمراجعة المسبقة
sys.setrecursionlimit(10000)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["DISABLE_SAFETY_FILTER"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print("تم ضبط حدود البيئة، إلغاء التخزين المؤقت، وتعطيل فلاتر الفحص المسبق بنجاح.")


سحب كود المشروع من GitHub (Ttbik)

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

# استدعاء مفتاح المفاتيح
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    # السحب المباشر بدون تحديد اسم فرع لتفادي خطأ الفرع المفقود
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), "خطأ: لم يتم العثور على ملفات المشروع"
sys.path.insert(0, CODE_DIR)
print(f"كود المشروع جاهز في المسار: {CODE_DIR}")


إعداد جارف الصور الشامل وتمرير كافة التباينات

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"جهاز المعالجة المتاح: {device}")

# تحويل الصور إلى 3 قنوات ألوان (RGB) بحجم 128x128
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("جاري تحميل مجموعة بيانات الصور الخام المضمونة...")

# تحميل مجموعة بيانات صور قياسية ومباشرة لتفادي مشاكل ملفات الفيديو
train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)

# إنشاء محمل البيانات بشكل آمن ومباشر
dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=False)
print(f"تم تجهيز محمل البيانات الشامل للصور بنجاح! عدد الحزم: {len(dataloader)}")


بناء أداة الترميز (Image Tokenizer / VAE)، التدريب، وحفظ image_tokenizer.pt

In [ ]:
class UncensoredImageTokenizer(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        # Encoders: ضغط الصورة من RGB إلى Latent Space
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),   # 64x64
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),  # 32x32
            nn.LeakyReLU(0.2),
            nn.Conv2d(128, embed_dim, kernel_size=3, padding=1)
        )
        # Decoders: إعادة بناء الصورة من Latent Space
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 128, kernel_size=4, stride=2, padding=1), # 64x64
            nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # 128x128
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return z, out

model = UncensoredImageTokenizer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
criterion = nn.L1Loss()

print("بدء تدريب أداة ترميز الصور مع تمرير كامل البيانات...")
model.train()
epochs = 2  # دورات سريعة لإخراج الملف فوراً

for epoch in range(epochs):
    running_loss = 0.0
    for batch_idx, (imgs, _) in enumerate(dataloader):
        imgs = imgs.to(device)
        optimizer.zero_grad()
        _, reconstructed = model(imgs)
        loss = criterion(reconstructed, imgs)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
        # التوقف المبكر بعد 100 حزمة لسرعة إنتاج الملف
        if batch_idx >= 100:
            break
    
    print(f"الدورة [{epoch+1}/{epochs}] - نسبة الخطأ (Loss): {running_loss/100:.5f}")

# حفظ الوزن المطلوب لدفتر الفيديو ودفتر الجدولة
output_pt_path = "/kaggle/working/image_tokenizer.pt"
torch.save(model.state_dict(), output_pt_path)

assert os.path.exists(output_pt_path), "خطأ: لم يتم حفظ الملف بشكل صحيح!"
print(f"\nتم بنجاح حفظ الملف المطلوب: {output_pt_path}")
print("يمكنك الآن حفظ الدفتر (Save Version -> Save & Run All) وربطه مع دفتر الفيديو ودفتر الجدولة التلقائية!")
